<a href="https://colab.research.google.com/github/albea47/DENG_repo1/blob/main/rd_portfolio_nsga2_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# NSGA-II R&D Portfolio Optimizer (Tri-objective) with Monte Carlo Uncertainty
# =============================================================================
# by Rafael Albea
#
# Purpose:
#   This script solves a constrained, multi-period R&D portfolio selection
#   problem using a Multi-Objective Genetic Algorithm (NSGA-II) combined with
#   Monte Carlo simulation to account for uncertainty in value and risk.
#
# High-level workflow:
#   1) Input handling
#      - Upload an Excel file interactively when running in Google Colab,
#        or read a file path when running locally.
#
#   2) Constraint & uncertainty model loading (from "Constraints & Goals" tab)
#      - Annual maximum budget constraints (MEUR) for Years 1–5
#      - Annual minimum budget constraints (MEUR) for Years 1–5
#      - Annual maximum FTE capacity constraints for Years 1–5
#      - Annual minimum FTE capacity constraints for Years 1–5
#      - Minimum portfolio investment share in Innovation projects
#      - Monte Carlo volatility parameters by project category:
#           * Non-Innovation NPV volatility
#           * Non-Innovation Risk volatility
#           * Innovation NPV volatility
#           * Innovation Risk volatility
#
#   3) Portfolio data ingestion
#      - Automatically detects the R&D program sheet and header depth
#      - Loads only valid R&D programs (numeric IDs, non-empty activity)
#      - Supports an arbitrary number of projects and multi-year profiles
#
#   4) Monte Carlo uncertainty modeling
#      - Applies category-dependent baseline volatility (Innovation vs Non-Innovation)
#      - Applies size-dependent volatility amplification:
#           * Larger projects exhibit higher tail dispersion
#           * Introduces diversification effects at portfolio level
#      - Uses lognormal multiplicative shocks with E[multiplier] = 1
#
#   5) Multi-objective optimization (DEAP / NSGA-II)
#      - Objectives (tri-objective):
#           * Maximize expected portfolio NPV (Monte Carlo mean)
#           * Minimize expected portfolio Risk (Monte Carlo mean)
#           * Minimize deterministic slack (unused budget + unused FTE)
#      - Hard constraints enforced via penalties:
#           * Annual budget min/max compliance
#           * Annual FTE min/max compliance
#           * Minimum Innovation investment share
#           * Project dependency satisfaction
#           * Mandatory project selection
#
#   6) Result export to Excel (XlsxWriter)
#      - "Scenario Results":
#           * Deterministic KPIs, feasibility diagnostics, and selection matrix
#           * Pareto front scatter chart (Risk vs NPV)
#      - "Project Contribution":
#           * Selection frequency of each R&D program
#      - "Innovation vs Efficiency":
#           * Innovation share vs efficiency (NPV / Risk)
#      - "Monte Carlo Analysis":
#           * Expected and tail statistics (P5 / P95)
#           * Pessimistic efficiency (NPV P5 / Risk P95)
#           * Scatter chart comparing expected, pessimistic, and optimistic outcomes
#           * Rows sorted by pessimistic efficiency for easier interpretation
#
# Notes:
#   - Designed to be scalable to large R&D portfolios and multi-year horizons
#   - Uses XlsxWriter for professional-quality Excel outputs and charts
# =============================================================================

# ---------------------------
# 0) Ensure packages
# ---------------------------
import sys, subprocess
def ensure_pkg(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"{pkg} installed.")

ensure_pkg("deap")
ensure_pkg("openpyxl")
ensure_pkg("XlsxWriter", "xlsxwriter")

# ---------------------------
# 1) Imports
# ---------------------------
import os, random
from typing import List, Dict, Optional, Tuple
import numpy as np
from deap import base, creator, tools, algorithms
from openpyxl import load_workbook
import xlsxwriter

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---------------------------
# 2) Global settings (defaults; overwritten from Excel when available)
# ---------------------------
MAX_BUDGETS       = [0.0]*5
MAX_FTE           = [0.0]*5
MIN_FTE_REQ       = [0.0]*5
MIN_BUDGET_REQ    = [0.0]*5
MIN_INNOVATION_PERCENTAGE = 0.0

# Category volatilities (defaults; overwritten from Excel when available)
NPV_VOL_BASE   = 0.15
RISK_VOL_BASE  = 0.10
NPV_VOL_INNOV  = 0.30
RISK_VOL_INNOV = 0.20

# Size-dependent volatility amplification (standard behavior)
ALPHA_SIZE_NPV  = 0.80
ALPHA_SIZE_RISK = 1.20

# Safety cap for volatility (prevents extreme sigmas if inputs are unusual)
VOL_CAP = 0.95

# GA hyperparameters
POP_SIZE = 432
NGEN     = 180
CXPB     = 0.90
MUTPB    = 0.20

# Monte Carlo settings
N_TRIALS = 300

# Global data containers
portfolio_data: List[Dict] = []
NPV_BASE: Optional[np.ndarray]  = None
RISK_BASE: Optional[np.ndarray] = None
MC_NPV_MUL: Optional[np.ndarray]  = None
MC_RISK_MUL: Optional[np.ndarray] = None


# =============================================================================
# 3) Excel parsing helpers (auto-detect project sheet & columns)
# =============================================================================
def _norm(s):
    return "" if s is None else str(s).replace("\n"," ").replace("\r"," ").strip()

def _norm_up(s):
    return _norm(s).upper()

def _composite_headers(ws, header_last_row:int)->List[str]:
    """Combine header rows 1..header_last_row into one composite header per column."""
    last_col = ws.max_column
    base = [ws.cell(row=header_last_row, column=c).value for c in range(1,last_col+1)]
    while base and base[-1] is None:
        base.pop()
    width = len(base)
    out=[]
    for c in range(1, width+1):
        parts=[]
        for r in range(1, header_last_row+1):
            v = ws.cell(row=r, column=c).value
            if v is not None and str(v).strip()!="":
                parts.append(_norm(v))
        out.append(" ".join(parts))
    return out

def _find_block_years(headers_up:List[str], block_label:str)->List[int]:
    """Find 5 year columns for a block label (INVESTMENT/FTE). Returns 0-based indices."""
    idx=[]
    for k in range(1,6):
        token=f"YEAR {k}"
        found=None
        for i,h in enumerate(headers_up):
            if block_label in h and token in h and i not in idx:
                found=i; break
        if found is not None:
            idx.append(found)
    return sorted(idx) if len(idx)==5 else []

def _try_detect_columns(headers_up:List[str]):
    """Detect key columns from composite headers."""
    def find_col_exact(names:List[str]):
        names=[n.upper() for n in names]
        for i,h in enumerate(headers_up):
            if any(h==n for n in names): return i
        return None

    def find_col_contains(names:List[str]):
        names=[n.upper() for n in names]
        for i,h in enumerate(headers_up):
            if any(n in h for n in names): return i
        return None

    id_col   = find_col_exact(["PROJECT #","ID","#ID"]) or find_col_contains(["PROJECT #","PROJECT NUMBER","ID","#ID"])
    npv_col  = find_col_contains(["NPV"])
    risk_col = find_col_contains(["RISK"])
    cat_col  = find_col_contains(["CATEGORY"])
    dep_col  = find_col_contains(["DEPENDENCY"])
    must_col = find_col_contains(["MUST"])

    # If MUST is labeled with "Yes (1), No (0)" use the rightmost such column
    if must_col is None:
        yesno = [i for i,h in enumerate(headers_up) if "YES (1)" in h and "NO (0)" in h]
        if yesno:
            must_col = max(yesno)

    inv_idxs = _find_block_years(headers_up, "INVESTMENT")
    fte_idxs = _find_block_years(headers_up, "FTE")

    # fallback: generic YEAR columns (first 5 = investment, next 5 = fte)
    if len(inv_idxs)!=5 or len(fte_idxs)!=5:
        year_cols = [i for i,h in enumerate(headers_up) if "YEAR " in h]
        if len(year_cols)>=10:
            if len(inv_idxs)!=5: inv_idxs = year_cols[:5]
            if len(fte_idxs)!=5: fte_idxs = year_cols[5:10]

    missing=[]
    if npv_col is None:  missing.append("NPV")
    if risk_col is None: missing.append("RISK")
    if cat_col is None:  missing.append("CATEGORY")
    if len(inv_idxs)!=5: missing.append("INVESTMENT Y1..Y5")
    if len(fte_idxs)!=5: missing.append("FTE Y1..Y5")

    ok = (len(missing)==0)
    cols = (id_col, npv_col, risk_col, cat_col, dep_col, must_col, inv_idxs, fte_idxs)
    return ok, missing, cols

def _auto_detect_start_row(ws, max_header_depth:int=25):
    """Try header_last_row=1..max_header_depth until we can detect all needed columns."""
    for header_last_row in range(1, max_header_depth+1):
        headers = _composite_headers(ws, header_last_row)
        headers_up = [_norm_up(h) for h in headers]
        ok, _, cols = _try_detect_columns(headers_up)
        if ok:
            return header_last_row + 1, headers, headers_up, cols
    return None, None, None, None

def _auto_detect_end_row(ws, start_row:int, headers_len:int)->int:
    """Scan down until we encounter a long blank streak; last_data_row is end."""
    max_row = ws.max_row
    blank_streak = 0
    BLANK_STREAK_STOP = 8
    last_data_row = start_row - 1
    for r in range(start_row, max_row + 1):
        row_vals = [ws.cell(row=r, column=c+1).value for c in range(headers_len)]
        if any(v not in (None,"") and str(v).strip() != "" for v in row_vals):
            last_data_row = r
            blank_streak = 0
        else:
            blank_streak += 1
            if blank_streak >= BLANK_STREAK_STOP:
                break
    return last_data_row if last_data_row >= start_row else start_row - 1

def find_projects_sheet(wb, exclude=("Constraints & Goals","Constraints","Goals")):
    """Select the most plausible project sheet: first sheet with recognizable headers."""
    for name in wb.sheetnames:
        if any(name.strip().lower()==ex.lower() for ex in exclude):
            continue
        ws = wb[name]
        start_row, headers, headers_up, cols = _auto_detect_start_row(ws)
        if start_row is not None:
            print(f"[INFO] Project sheet detected: '{name}' (data starts at row {start_row})")
            return ws, start_row, headers, headers_up, cols
    # fallback try all
    for name in wb.sheetnames:
        ws = wb[name]
        start_row, headers, headers_up, cols = _auto_detect_start_row(ws)
        if start_row is not None:
            print(f"[INFO] Project sheet detected (fallback): '{name}' (data starts at row {start_row})")
            return ws, start_row, headers, headers_up, cols
    raise ValueError("No projects sheet found with recognizable headers.")

def load_portfolio_from_excel_auto(path:str, sheet_name:Optional[str]=None)->List[Dict]:
    """Load programs table with auto-detected sheet and header depth. Skips blank/invalid IDs."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Input Excel not found: '{path}'")

    wb = load_workbook(path, data_only=True)

    if sheet_name is None:
        ws, start_row, headers, headers_up, cols = find_projects_sheet(wb)
    else:
        ws = wb[sheet_name]
        start_row, headers, headers_up, cols = _auto_detect_start_row(ws)
        if start_row is None:
            raise ValueError(f"Could not detect headers in sheet '{sheet_name}'.")

    (id_col, npv_col, risk_col, cat_col, dep_col, must_col, inv_idxs, fte_idxs) = cols

    end_row = _auto_detect_end_row(ws, start_row, len(headers))
    if end_row < start_row:
        return []

    def fnum(v, d=0.0):
        try:
            if v is None or (isinstance(v,str) and v.strip()==""): return d
            return float(v)
        except:
            return d

    def fint_or_str(v):
        if v is None or (isinstance(v,str) and v.strip()==""):
            return None
        try:
            return int(float(v))
        except:
            return str(v).strip()

    def _valid_id(v):
        if v is None: return False
        s = str(v).strip()
        if s == "": return False
        # protect against accidental "AUTO1" etc.
        # accept numeric IDs only (common in your dataset)
        try:
            float(s)
            return True
        except:
            return False

    rows=[]
    for r in range(start_row, end_row+1):
        vals = [ws.cell(row=r, column=c+1).value for c in range(len(headers))]
        rid_raw = vals[id_col] if id_col is not None else None
        if not _valid_id(rid_raw):
            continue

        rid = int(float(rid_raw))

        npv  = fnum(vals[npv_col], 0.0)
        risk = fnum(vals[risk_col], 0.0)
        cat  = str(vals[cat_col]).strip() if vals[cat_col] not in (None,"") else "Innovation"
        inv  = [fnum(vals[i], 0.0) for i in inv_idxs]
        fte  = [fnum(vals[i], 0.0) for i in fte_idxs]
        dep  = fint_or_str(vals[dep_col]) if dep_col is not None else None

        must = 0
        if must_col is not None and vals[must_col] not in (None,""):
            try:
                must = int(float(vals[must_col]))
            except:
                must = 0

        has_activity = (
            (npv>0.0) or (risk>0.0) or any(v>0.0 for v in inv) or any(v>0.0 for v in fte) or (must==1)
        )
        if has_activity:
            rows.append({
                "ID": rid, "NPV": npv, "RISK": risk, "CATEGORY": cat,
                "INVESTMENT": inv, "FTE": fte,
                "DEPENDENCY_ID": dep, "MUST_BE_SELECTED": must
            })

    # Make DISPLAY_ID stable if duplicates exist
    seen={}
    for rec in rows:
        disp = str(rec["ID"])
        if disp in seen:
            seen[disp]+=1
            rec["DISPLAY_ID"]=f"{disp}-{seen[disp]}"
        else:
            seen[disp]=1
            rec["DISPLAY_ID"]=disp

    return rows


# =============================================================================
# 4) Load constraints from "Constraints & Goals" (including volatilities)
# =============================================================================
def load_constraints_from_excel(path: str,
                                sheet_name: str = "Constraints & Goals"):
    """
    Returns:
      (max_bud[5], max_fte[5], min_fte[5], min_bud[5], innov_pct,
       npv_vol_noninnov, risk_vol_noninnov, npv_vol_innov, risk_vol_innov)
    """
    default_budgets = [0,0,0,0,0]
    default_max_fte = [0,0,0,0,0]
    default_min_fte = [0,0,0,0,0]
    default_min_bud = [0,0,0,0,0]
    default_innov   = 0.0

    default_npv_non = NPV_VOL_BASE
    default_rsk_non = RISK_VOL_BASE
    default_npv_inn = NPV_VOL_INNOV
    default_rsk_inn = RISK_VOL_INNOV

    wb = load_workbook(path, data_only=True)
    if sheet_name not in wb.sheetnames:
        print(f"[WARN] Sheet '{sheet_name}' not found. Using defaults.")
        return (default_budgets, default_max_fte, default_min_fte, default_min_bud, default_innov,
                default_npv_non, default_rsk_non, default_npv_inn, default_rsk_inn)

    ws = wb[sheet_name]

    def row_label_and_col(r):
        for c in range(1, ws.max_column + 1):
            v = ws.cell(row=r, column=c).value
            if isinstance(v, str) and v.strip():
                return v.strip(), c
        return None, None

    def to_float_excel(v):
        if v is None:
            return None
        if isinstance(v, (int, float)):
            return float(v)
        if isinstance(v, str):
            s = v.strip().replace("%", "")
            if s == "":
                return None
            try:
                return float(s)
            except:
                return None
        return None

    def next_n_nums(r, start_col=1, n=5):
        vals = []
        for c in range(start_col, ws.max_column + 1):
            v = ws.cell(row=r, column=c).value
            num = to_float_excel(v)
            if num is None:
                continue
            vals.append(num)
            if len(vals) == n:
                break
        return vals

    max_bud=None; max_fte=None; min_fte=None; min_bud=None; innov=None
    npv_non=default_npv_non; rsk_non=default_rsk_non; npv_inn=default_npv_inn; rsk_inn=default_rsk_inn

    for r in range(1, ws.max_row + 1):
        lbl, lbl_col = row_label_and_col(r)
        if not lbl:
            continue
        u = lbl.lower()

        if max_bud is None and "max annual budget" in u:
            cand = next_n_nums(r, 4, 5)
            if len(cand) == 5:
                max_bud = cand

        if max_fte is None and (("max annual capacity" in u) or ("annual max capacity" in u)) and "fte" in u:
            cand = next_n_nums(r, 4, 5)
            if len(cand) == 5:
                max_fte = cand

        if min_fte is None and (("min annual capacity" in u) or ("annual min capacity" in u)) and "fte" in u:
            cand = next_n_nums(r, 4, 5)
            if len(cand) == 5:
                min_fte = cand

        if min_bud is None and "min annual budget" in u:
            cand = next_n_nums(r, 4, 5)
            if len(cand) == 5:
                min_bud = cand

        if innov is None and ("innovation" in u and "min" in u and "%" in u):
            cand = next_n_nums(r, start_col=lbl_col + 1, n=1)
            if cand:
                v = cand[0]
                innov = v / 100.0 if v > 1.0 else v

        if "non-innovation" in u and "npv" in u and "vol" in u:
            cand = next_n_nums(r, 4, 1)
            if cand:
                v = cand[0]
                npv_non = v / 100.0 if v > 1.0 else v

        if "non-innovation" in u and "risk" in u and "vol" in u:
            cand = next_n_nums(r, 4, 1)
            if cand:
                v = cand[0]
                rsk_non = v / 100.0 if v > 1.0 else v

        if "innovation" in u and "npv" in u and "vol" in u and "non-innovation" not in u:
            cand = next_n_nums(r, 4, 1)
            if cand:
                v = cand[0]
                npv_inn = v / 100.0 if v > 1.0 else v

        if "innovation" in u and "risk" in u and "vol" in u and "non-innovation" not in u:
            cand = next_n_nums(r, 4, 1)
            if cand:
                v = cand[0]
                rsk_inn = v / 100.0 if v > 1.0 else v

    max_bud = max_bud if max_bud else default_budgets
    max_fte = max_fte if max_fte else default_max_fte
    min_fte = min_fte if min_fte else default_min_fte
    min_bud = min_bud if min_bud else default_min_bud
    innov   = innov if innov is not None else default_innov

    print("[INFO] Constraints loaded:")
    print(f"  MAX_BUDGETS={max_bud}")
    print(f"  MAX_FTE={max_fte}")
    print(f"  MIN_FTE_REQ={min_fte}")
    print(f"  MIN_BUDGET_REQ={min_bud}")
    print(f"  MIN_INNOVATION_PERCENTAGE={innov}")
    print("[INFO] Volatility loaded:")
    print(f"  NON-INNOV: NPV_VOL={npv_non}, RISK_VOL={rsk_non}")
    print(f"  INNOV:     NPV_VOL={npv_inn}, RISK_VOL={rsk_inn}")

    return (max_bud, max_fte, min_fte, min_bud, innov, npv_non, rsk_non, npv_inn, rsk_inn)


# =============================================================================
# 5) Monte Carlo initialization (category-dependent volatility)
# =============================================================================
def init_monte_carlo(n_projects:int):
    """
    Precompute Monte Carlo multipliers for NPV and Risk using:
      1) Category-dependent baseline volatility (Innovation vs Non-Innovation)
      2) Size-dependent amplification:
            sigma_i = sigma_cat(i) * (1 + alpha * normalized_size_i)

    This makes the Monte Carlo pessimistic ranking potentially differ from the
    deterministic ranking by introducing diversification effects (large bets
    have more tail dispersion) without adding correlations or changing the
    meaning of the Risk input.
    """
    global NPV_BASE, RISK_BASE, MC_NPV_MUL, MC_RISK_MUL
    global NPV_VOL_BASE, RISK_VOL_BASE, NPV_VOL_INNOV, RISK_VOL_INNOV
    global ALPHA_SIZE_NPV, ALPHA_SIZE_RISK, VOL_CAP

    NPV_BASE  = np.array([p['NPV']  for p in portfolio_data], dtype=float)
    RISK_BASE = np.array([p['RISK'] for p in portfolio_data], dtype=float)

    # (1) baseline category volatility vectors
    vol_npv  = np.full(n_projects, NPV_VOL_BASE,  dtype=float)
    vol_risk = np.full(n_projects, RISK_VOL_BASE, dtype=float)

    for i, p in enumerate(portfolio_data):
        if str(p.get('CATEGORY','')).strip().lower() == "innovation":
            vol_npv[i]  = NPV_VOL_INNOV
            vol_risk[i] = RISK_VOL_INNOV

    # (2) normalized project size measures in [0, 1]
    max_npv = float(np.max(NPV_BASE)) if np.max(NPV_BASE) > 0 else 1.0
    max_rsk = float(np.max(RISK_BASE)) if np.max(RISK_BASE) > 0 else 1.0
    s_npv   = NPV_BASE  / max_npv
    s_risk  = RISK_BASE / max_rsk

    # (3) apply size-dependent amplification
    vol_npv  = vol_npv  * (1.0 + ALPHA_SIZE_NPV  * s_npv)
    vol_risk = vol_risk * (1.0 + ALPHA_SIZE_RISK * s_risk)

    # Safety cap
    vol_npv  = np.clip(vol_npv,  0.0, VOL_CAP)
    vol_risk = np.clip(vol_risk, 0.0, VOL_CAP)

    # (4) convert desired coefficient of variation to lognormal parameters
    # Ensures E[multiplier] = 1 by setting mu = -0.5*sigma^2
    sigma_npv  = np.sqrt(np.log(1.0 + vol_npv**2))
    mu_npv     = -0.5 * sigma_npv**2
    sigma_risk = np.sqrt(np.log(1.0 + vol_risk**2))
    mu_risk    = -0.5 * sigma_risk**2

    # (5) sample multipliers (reproducible)
    np.random.seed(123)
    z_npv  = np.random.normal(size=(N_TRIALS, n_projects))
    z_risk = np.random.normal(size=(N_TRIALS, n_projects))

    MC_NPV_MUL  = np.exp(mu_npv  + sigma_npv  * z_npv)
    MC_RISK_MUL = np.exp(mu_risk + sigma_risk * z_risk)

# =============================================================================
# 6) Portfolio math + constraints
# =============================================================================
def _id_key(v):
    try:
        return int(float(v))
    except Exception:
        return str(v).strip()

def totals_for_solution(ind):
    total_npv=0.0
    total_risk=0.0
    inv=[0.0]*5
    fte=[0.0]*5
    total_inv=0.0
    total_innov=0.0

    for g,p in zip(ind, portfolio_data):
        if g:
            total_npv += p['NPV']
            total_risk += p['RISK']
            for y in range(5):
                inv[y]+=p['INVESTMENT'][y]
                fte[y]+=p['FTE'][y]
                total_inv += p['INVESTMENT'][y]
            if str(p['CATEGORY']).strip().lower()=="innovation":
                total_innov += sum(p['INVESTMENT'])

    budget_slack = sum(max(0.0, MAX_BUDGETS[y] - inv[y]) for y in range(5))
    fte_slack    = sum(max(0.0, MAX_FTE[y]     - fte[y]) for y in range(5))
    total_slack  = budget_slack + fte_slack

    return (total_npv, total_risk, inv, fte, total_inv, total_innov, budget_slack, fte_slack, total_slack)

def feasibility_and_penalty(ind):
    _, _, inv, fte, total_inv, total_innov, *_ = totals_for_solution(ind)
    pen = 0.0

    for y in range(5):
        if inv[y] > MAX_BUDGETS[y]:
            pen += (inv[y]-MAX_BUDGETS[y]) * 10.0
        if MIN_BUDGET_REQ[y] > 0.0 and inv[y] < MIN_BUDGET_REQ[y]:
            pen += (MIN_BUDGET_REQ[y]-inv[y]) * 5.0

        if fte[y] > MAX_FTE[y]:
            pen += (fte[y]-MAX_FTE[y]) * 1.0
        if MIN_FTE_REQ[y] > 0.0 and fte[y] < MIN_FTE_REQ[y]:
            pen += (MIN_FTE_REQ[y]-fte[y]) * 1.0

    if total_inv <= 1e-12:
        pen += 1000.0

    if total_inv > 1e-12 and MIN_INNOVATION_PERCENTAGE > 0.0:
        innov_min = MIN_INNOVATION_PERCENTAGE * total_inv
        if total_innov < innov_min:
            pen += (innov_min-total_innov) * 5.0

    id_to_idx = {_id_key(p['ID']): i for i,p in enumerate(portfolio_data)}
    for i,(g,p) in enumerate(zip(ind, portfolio_data)):
        if g and p['DEPENDENCY_ID'] is not None:
            dep_idx = id_to_idx.get(_id_key(p['DEPENDENCY_ID']))
            if dep_idx is not None and ind[dep_idx]==0:
                pen += 100.0

    for g,p in zip(ind, portfolio_data):
        if p['MUST_BE_SELECTED']==1 and g==0:
            pen += 200.0

    return (pen==0.0), pen


def scenario_diagnostics(ind):
    npv, risk, inv, fte, total_inv, total_innov, budget_slack, fte_slack, total_slack = totals_for_solution(ind)
    eff = (npv/risk) if risk > 1e-12 else float('inf')

    innov_pct = (total_innov/total_inv) if total_inv > 1e-12 else 0.0
    budget_max_ok = all(inv[y] <= MAX_BUDGETS[y] for y in range(5))
    budget_min_ok = all((inv[y] >= MIN_BUDGET_REQ[y]) if MIN_BUDGET_REQ[y]>0 else True for y in range(5))
    fte_max_ok    = all(fte[y] <= MAX_FTE[y] for y in range(5))
    fte_min_ok    = all((fte[y] >= MIN_FTE_REQ[y]) if MIN_FTE_REQ[y]>0 else True for y in range(5))
    innov_ok      = (MIN_INNOVATION_PERCENTAGE <= 0.0) or (innov_pct >= MIN_INNOVATION_PERCENTAGE)

    return {
        "npv": npv, "risk": risk, "eff": eff,
        "budget_slack": budget_slack, "fte_slack": fte_slack, "total_slack": total_slack,
        "inv_years": inv, "fte_years": fte,
        "innov_pct": innov_pct,
        "budget_max_ok": budget_max_ok, "budget_min_ok": budget_min_ok,
        "fte_max_ok": fte_max_ok, "fte_min_ok": fte_min_ok,
        "innovation_ok": innov_ok
    }

def scenario_mc_stats(ind):
    sel = np.array(ind, dtype=float)
    npv_trials  = (MC_NPV_MUL  * NPV_BASE)  @ sel
    risk_trials = (MC_RISK_MUL * RISK_BASE) @ sel

    mean_npv = float(np.mean(npv_trials))
    std_npv  = float(np.std(npv_trials, ddof=1))
    p5_npv, p95_npv = np.percentile(npv_trials, [5,95])

    mean_risk = float(np.mean(risk_trials))
    std_risk  = float(np.std(risk_trials, ddof=1))
    p5_risk, p95_risk = np.percentile(risk_trials, [5,95])

    return {
        "mean_npv": mean_npv, "std_npv": std_npv, "p5_npv": float(p5_npv), "p95_npv": float(p95_npv),
        "mean_risk": mean_risk, "std_risk": std_risk, "p5_risk": float(p5_risk), "p95_risk": float(p95_risk),
    }


# =============================================================================
# 7) DEAP NSGA-II setup (tri-objective)
# =============================================================================
def setup_deap(n_projects:int):
    try:
        creator.Fitness3
    except AttributeError:
        creator.create("Fitness3", base.Fitness, weights=(1.0, -1.0, -1.0))  # max npv, min risk, min slack
    try:
        creator.Individual3
    except AttributeError:
        creator.create("Individual3", list, fitness=creator.Fitness3)

    tb = base.Toolbox()
    random.seed(42)

    tb.register("attr_bool", random.randint, 0, 1)
    tb.register("individual", tools.initRepeat, creator.Individual3, tb.attr_bool, n_projects)
    tb.register("population", tools.initRepeat, list, tb.individual)

    def evaluate(ind):
        # deterministic slack + penalties
        total_slack = totals_for_solution(ind)[-1]
        feasible, pen = feasibility_and_penalty(ind)

        # Monte Carlo expected NPV and expected Risk
        sel = np.array(ind, dtype=float)
        npv_trials  = (MC_NPV_MUL  * NPV_BASE)  @ sel
        risk_trials = (MC_RISK_MUL * RISK_BASE) @ sel
        mean_npv  = float(np.mean(npv_trials))
        mean_risk = float(np.mean(risk_trials))

        if not feasible:
            mean_npv   -= 1_000.0 * pen
            mean_risk  += 1_000.0 * pen
            total_slack+= 1_000.0 * pen

        return (mean_npv, mean_risk, total_slack)

    tb.register("evaluate", evaluate)
    tb.register("mate", tools.cxTwoPoint)
    tb.register("mutate", tools.mutFlipBit, indpb=1.0/n_projects)
    tb.register("select", tools.selNSGA2)
    return tb

def run_nsga2(tb, pop_size=POP_SIZE, ngen=NGEN, cxpb=CXPB, mutpb=MUTPB):
    pop = tb.population(pop_size)
    for ind in pop:
        ind.fitness.values = tb.evaluate(ind)
    pop = tb.select(pop, len(pop))

    archive = tools.ParetoFront(similar=lambda a,b: a==b)

    for _ in range(ngen):
        offspring = algorithms.varAnd(pop, tb, cxpb=cxpb, mutpb=mutpb)
        for ind in offspring:
            ind.fitness.values = tb.evaluate(ind)
        pop = tb.select(pop + offspring, pop_size)
        archive.update(pop)

    feasible=[]
    for ind in archive:
        ok,_ = feasibility_and_penalty(ind)
        if ok:
            feasible.append(ind)

    uniq, seen = [], set()
    for ind in feasible:
        key = tuple(ind)
        if key not in seen:
            uniq.append(ind)
            seen.add(key)
    return uniq


# =============================================================================
# 8) Excel export (NO rank columns)
# =============================================================================
def export_to_excel_xlsxwriter(sorted_inds: List, output_path:str):
    workbook  = xlsxwriter.Workbook(output_path)

    fmt_header = workbook.add_format({'bold': True})
    fmt_num    = workbook.add_format({'num_format': '0.000'})
    fmt_pct    = workbook.add_format({'num_format': '0.0%'})
    fmt_int    = workbook.add_format({'num_format': '0'})

    # ---- Scenario Results
    ws = workbook.add_worksheet("Scenario Results")

    prog_labels = [f"ID[{p['DISPLAY_ID']}]" for p in portfolio_data]
    header = (
        ["Scenario ID",
         "Total NPV (MEUR)", "Total Risk score",
         "Budget Slack", "FTE Slack", "Total Slack",
         "Efficiency (NPV/Risk)"] +
        [f"Invest Y{y}" for y in range(1,6)] +
        [f"FTE Y{y}"    for y in range(1,6)] +
        ["Innovation %",
         "Budget Max OK", "Budget Min OK",
         "FTE Max OK", "FTE Min OK", "Innovation OK"] +
        prog_labels
    )
    for c,h in enumerate(header):
        ws.write(0, c, h, fmt_header)

    diag_cache=[]
    mc_cache=[]
    for r, ind in enumerate(sorted_inds, start=1):
        d  = scenario_diagnostics(ind)
        mc = scenario_mc_stats(ind)
        diag_cache.append((ind,d))
        mc_cache.append(mc)

        row = [
            r,
            d["npv"], d["risk"],
            d["budget_slack"], d["fte_slack"], d["total_slack"],
            (d["eff"] if d["eff"]!=float('inf') else 0.0)
        ]
        row += d["inv_years"]
        row += d["fte_years"]
        row += [d["innov_pct"]]
        row += [int(d["budget_max_ok"]), int(d["budget_min_ok"]),
                int(d["fte_max_ok"]), int(d["fte_min_ok"]), int(d["innovation_ok"])]
        row += [int(g) for g in ind]

        for c,val in enumerate(row):
            h = header[c]
            if h == "Scenario ID" or h.endswith("OK") or h.startswith("ID["):
                ws.write_number(r, c, val, fmt_int)
            elif h == "Innovation %":
                ws.write_number(r, c, val, fmt_pct)
            else:
                ws.write_number(r, c, val, fmt_num)

    last_row = len(sorted_inds) + 1
    x_rng = f"='Scenario Results'!$B$2:$B${last_row}"
    y_rng = f"='Scenario Results'!$C$2:$C${last_row}"

    chart = workbook.add_chart({'type': 'scatter'})
    chart.add_series({
        'name': 'Deterministic Pareto',
        'categories': x_rng,
        'values':     y_rng,
        'marker': {'type':'circle','size':6,
                   'border':{'color':'black'},
                   'fill':{'color':'black'}},
        'line':   {'none': True},
    })
    chart.set_title({'name':'Deterministic Pareto Front (Risk vs. Value)',
                     'name_font': {'name':'Times New Roman', 'bold': True}})
    chart.set_x_axis({'name':'Total NPV (MEUR)',
                      'name_font':{'name':'Times New Roman'},
                      'num_font': {'name':'Times New Roman'},
                      'major_gridlines': {'visible': True}})
    chart.set_y_axis({'name':'Total Risk score',
                      'name_font':{'name':'Times New Roman'},
                      'num_font': {'name':'Times New Roman'},
                      'major_gridlines': {'visible': True}})
    chart.set_legend({'none': True})
    ws.insert_chart('H2', chart, {'x_scale':1.1,'y_scale':1.1})

    # ---- Project Contribution
    ws_c = workbook.add_worksheet("Project Contribution")
    ws_c.write(0,0,"Project",fmt_header)
    ws_c.write(0,1,"Selected Count",fmt_header)
    ws_c.write(0,2,"Selection Rate",fmt_header)

    counts = [0]*len(portfolio_data)
    for ind,_ in diag_cache:
        for i,b in enumerate(ind):
            if b==1:
                counts[i]+=1
    total_scenarios = max(1, len(sorted_inds))

    for i,p in enumerate(portfolio_data, start=1):
        ws_c.write(i,0, p["DISPLAY_ID"])
        ws_c.write(i,1, counts[i-1], fmt_int)
        ws_c.write_number(i,2, counts[i-1]/total_scenarios, fmt_pct)

    last_proj_row = len(portfolio_data) + 1
    cat = f"='Project Contribution'!$A$2:$A${last_proj_row}"
    val = f"='Project Contribution'!$B$2:$B${last_proj_row}"

    bchart = workbook.add_chart({'type':'column'})
    bchart.add_series({
        'name':'Selection Count',
        'categories': cat,
        'values':     val,
        'border': {'color':'black'},
        'fill':   {'color':'black'},
    })
    bchart.set_title({'name':'Project Selection Frequency',
                      'name_font': {'name':'Times New Roman','bold':True}})
    bchart.set_legend({'none': True})
    ws_c.insert_chart('E2', bchart, {'x_scale':1.3,'y_scale':1.2})

    # ---- Innovation vs Efficiency
    ws_ie = workbook.add_worksheet("Innovation vs Efficiency")
    ws_ie.write(0,0,"Scenario ID",fmt_header)
    ws_ie.write(0,1,"Efficiency",fmt_header)
    ws_ie.write(0,2,"Innovation %",fmt_header)

    for r, (_,d) in enumerate(diag_cache, start=1):
        ws_ie.write_number(r,0, r, fmt_int)
        ws_ie.write_number(r,1, d["eff"] if d["eff"]!=float('inf') else 0.0, fmt_num)
        ws_ie.write_number(r,2, d["innov_pct"], fmt_pct)

    last_ie = len(diag_cache) + 1
    x_rng_ie = f"='Innovation vs Efficiency'!$B$2:$B${last_ie}"
    y_rng_ie = f"='Innovation vs Efficiency'!$C$2:$C${last_ie}"

    ichart = workbook.add_chart({'type':'scatter'})
    ichart.add_series({
        'name':'Innovation vs Efficiency',
        'categories': x_rng_ie,
        'values':     y_rng_ie,
        'marker': {'type':'circle','size':6,
                   'border':{'color':'black'},
                   'fill':{'color':'black'}},
        'line': {'none': True}
    })
    ichart.set_title({'name':'Innovation Share vs Efficiency',
                      'name_font':{'name':'Times New Roman','bold':True}})
    ichart.set_x_axis({'name':'Efficiency (NPV / Risk)'})
    ichart.set_y_axis({'name':'Innovation Share', 'min':0, 'max':1})
    ichart.set_legend({'none': True})
    ws_ie.insert_chart('E2', ichart, {'x_scale':1.2,'y_scale':1.2})

    # ---- Monte Carlo Analysis (no ranks)
    ws_mc = workbook.add_worksheet("Monte Carlo Analysis")
    header_mc = [
        "Scenario ID",
        "MC NPV mean", "MC NPV std", "MC NPV P5", "MC NPV P95",
        "MC Risk mean", "MC Risk std", "MC Risk P5", "MC Risk P95",
        "Pessimistic Eff (NPV P5 / Risk P95)"
    ]
    for c,h in enumerate(header_mc):
        ws_mc.write(0, c, h, fmt_header)

    # Build rows with pessimistic efficiency
    mc_rows = []
    for scenario_id, mc in enumerate(mc_cache, start=1):
        pe = (mc["p5_npv"] / mc["p95_risk"]) if mc["p95_risk"] > 1e-12 else 0.0
        mc_rows.append((scenario_id, mc, pe))

    # Sort by pessimistic efficiency (descending)
    mc_rows.sort(key=lambda x: x[2], reverse=True)

    # Write sorted rows
    for excel_row, (scenario_id, mc, pe) in enumerate(mc_rows, start=1):
        row = [
            scenario_id,
            mc["mean_npv"], mc["std_npv"], mc["p5_npv"], mc["p95_npv"],
            mc["mean_risk"], mc["std_risk"], mc["p5_risk"], mc["p95_risk"],
            pe
        ]
        for c, val in enumerate(row):
            if c == 0:
                ws_mc.write_number(excel_row, c, val, fmt_int)
            else:
                ws_mc.write_number(excel_row, c, val, fmt_num)

    last_mc = len(mc_cache) + 1
    x_exp  = f"='Monte Carlo Analysis'!$B$2:$B${last_mc}"  # mean npv
    y_exp  = f"='Monte Carlo Analysis'!$F$2:$F${last_mc}"  # mean risk
    x_pess = f"='Monte Carlo Analysis'!$D$2:$D${last_mc}"  # p5 npv
    y_pess = f"='Monte Carlo Analysis'!$I$2:$I${last_mc}"  # p95 risk
    x_opt  = f"='Monte Carlo Analysis'!$E$2:$E${last_mc}"  # p95 npv
    y_opt  = f"='Monte Carlo Analysis'!$H$2:$H${last_mc}"  # p5 risk

    mc_chart = workbook.add_chart({'type':'scatter'})

    mc_chart.add_series({
        'name': 'Expected',
        'categories': x_exp,
        'values':     y_exp,
        'marker': {'type':'circle','size':6,
                   'border':{'color':'black'},
                   'fill':{'color':'black'}},
        'line': {'none': True}
    })
    mc_chart.add_series({
        'name': 'Pessimistic (P5 NPV / P95 Risk)',
        'categories': x_pess,
        'values':     y_pess,
        'marker': {'type':'x','size':7,
                   'border':{'color':'red'},
                   'fill':{'color':'red'}},
        'line': {'none': True}
    })
    mc_chart.add_series({
        'name': 'Optimistic (P95 NPV / P5 Risk)',
        'categories': x_opt,
        'values':     y_opt,
        'marker': {'type':'square','size':6,
                   'border':{'color':'green'},
                   'fill':{'color':'green'}},
        'line': {'none': True}
    })

    mc_chart.set_title({'name':'Monte Carlo Frontier – Expected vs Optimistic/Pessimistic'})
    mc_chart.set_x_axis({'name':'NPV (MEUR)'})
    mc_chart.set_y_axis({'name':'Risk'})
    mc_chart.set_legend({'position':'bottom'})
    ws_mc.insert_chart('L2', mc_chart, {'x_scale':1.1,'y_scale':1.1})

    workbook.close()


# =============================================================================
# 9) Main
# =============================================================================
def main():
    global MAX_BUDGETS, MAX_FTE, MIN_FTE_REQ, MIN_BUDGET_REQ, MIN_INNOVATION_PERCENTAGE
    global NPV_VOL_BASE, RISK_VOL_BASE, NPV_VOL_INNOV, RISK_VOL_INNOV
    global portfolio_data

    if IN_COLAB:
        from google.colab import files  # type: ignore
        print("Please upload the Excel file to use as input…")
        up = files.upload()
        if not isinstance(up, dict) or len(up)==0:
            raise RuntimeError("No file uploaded.")
        input_path = next(iter(up.keys()))
        print(f"Using uploaded file: {input_path}")
    else:
        if len(sys.argv) < 2:
            raise RuntimeError("Run locally as: python script.py <input.xlsx>")
        input_path = sys.argv[1]

    # Load constraints (includes volatilities)
    (MAX_BUDGETS, MAX_FTE, MIN_FTE_REQ, MIN_BUDGET_REQ, MIN_INNOVATION_PERCENTAGE,
     NPV_VOL_BASE, RISK_VOL_BASE, NPV_VOL_INNOV, RISK_VOL_INNOV) = load_constraints_from_excel(
        input_path, sheet_name="Constraints & Goals"
    )

    # Load portfolio
    portfolio_data = load_portfolio_from_excel_auto(input_path, sheet_name=None)
    n = len(portfolio_data)
    print(f"Detected {n} R&D programs (valid numeric IDs only).")
    if n == 0:
        raise RuntimeError("No projects detected. Check the project sheet headers/IDs.")

    # Init Monte Carlo (uses loaded volatilities)
    init_monte_carlo(n)

    # Run NSGA-II
    toolbox = setup_deap(n)
    pareto = run_nsga2(toolbox)

    if len(pareto) == 0:
        raise RuntimeError("No feasible Pareto scenarios found. Relax constraints or check inputs.")

    # Sort for presentation only
    def sort_key(ind):
        mc = scenario_mc_stats(ind)
        eff = mc["mean_npv"]/mc["mean_risk"] if mc["mean_risk"]>1e-12 else 0.0
        slack = totals_for_solution(ind)[-1]
        return (-eff, -mc["mean_npv"], mc["mean_risk"], slack)

    pareto_sorted = sorted(pareto, key=sort_key)

    out_path = os.path.join(os.getcwd(), "Portfolio_Scenarios_MC.xlsx")
    export_to_excel_xlsxwriter(pareto_sorted, out_path)
    print(f"Saved: {out_path}")

    if IN_COLAB:
        from google.colab import files  # type: ignore
        files.download(out_path)

if __name__ == "__main__":
    main()


Please upload the Excel file to use as input…


Saving R&D Dataset.xlsx to R&D Dataset (5).xlsx
Using uploaded file: R&D Dataset (5).xlsx
[INFO] Constraints loaded:
  MAX_BUDGETS=[28.0, 28.0, 29.0, 29.0, 30.0]
  MAX_FTE=[200.0, 200.0, 200.0, 200.0, 200.0]
  MIN_FTE_REQ=[160.0, 160.0, 160.0, 160.0, 160.0]
  MIN_BUDGET_REQ=[23.0, 23.0, 24.0, 24.0, 24.0]
  MIN_INNOVATION_PERCENTAGE=0.15
[INFO] Volatility loaded:
  NON-INNOV: NPV_VOL=0.15, RISK_VOL=0.1
  INNOV:     NPV_VOL=0.3, RISK_VOL=0.2
[INFO] Project sheet detected: 'R&D Program Dataset' (data starts at row 4)
Detected 30 R&D programs (valid numeric IDs only).
Saved: /content/Portfolio_Scenarios_MC.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>